In [1]:
# RAG(Retriever Augmentation Generation) 시스템 및 모델 파이프라인 구축(데이터 수집기 + Qdrant 의미 기반 검색엔진 적용)
# - Retrieval 단계: Qdrant 의미 기반 검색엔진을 통해 관련 문서를 빠르게 찾아냄
# - Augmentation 단계: 검색된 문서를 기반으로 LLM 입력 프롬프트를 강화 → 맥락 있는 답변 생성
# - Generation 단계: LLM이 최종 답변을 생성 → 단순 생성보다 정확성과 신뢰성이 높아짐
# - 예시) 구글, 네이버에서 사용되는 RAG 시스템 구현

# 학습 목표 - 실무에서 사용되는 파이프라인 이해 및 적용
# 1. 데이터 수집기
# - 수집 대상 도메인: Google News RSS
# - RSS 피드 파싱: feedparser 라이브러리로 RSS XML을 읽고 기사 항목 추출
# - 데이터 정제: title,link,description,pubDate,soure 필드 추출, pubDate -> Python datetime 변화
# - 수집 데이터 -> PostgreSQL 테이블 news_articles 에 Insert
# - DB 조회 + 로깅 설정으로 데이터 관리

# 2. Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

# 3. 임베딩 생성 후 Qdrant Collection에 Insert/Update
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 4. Qdrant 의미 기반 검색
# - 의미 기반 검색으로 관련 문서 조회

# 5. QA 처리
# - Qdrant 검색 결과 -> QA 토크나이저 + QA 모델
# - 형태소 분석으로 한국어 처리 강화
# - 문자열 -> 토큰 ID 변환, 예시: "AI는 의료 분야에서 활용된다." -> [101, 1234, 5678, ...]
# - 모델 입력: input_ids 토큰 ID 배열, attention_mask 패딩 여부 표시, 모델 출력을 텍스트로 복원 [101, 1234, 5678, ...] -> "AI는 의료 분야에서 활용된다."

# 6. 요약 처리
# - 요약 토크나이저 + 요약 모델 (KoBART 기반)
# - 반복 억제, 길이 조절, 다양성 확보 등 파라미터 튜닝
# - 후처리(clean_summary)로 중복 제거
# - from transformers import BartTokenizer # 영어 전용이라 맞지 않음

# 7. 최종 응답: 외부 서비스 연계 검토
# - Local LLM은 GPU 장비 한계로 로컬 실행은 패스 -> 대신 Copilot에 문의하여 자연스러운 답변 재구성
# - 검토: 현재 로컬 GPU 장비로는 생성형 LLM 모델을 도메인에 맞추어 파인튜닝은 불가능, 현재 추론 모델도 좋은 성능의 모델로 교체 불가능

# 8. 서비스: 구글, 네이버 RAG 시스템 구성
# - /llm_app/transformer_rag3_24_app.py
# - FastAPI 구동 정보: 터미널에서 구동, uvicorn transformer_rag3_24_app:app --reload, 경로 포함 uvicorn LLM.llm_app.transformer_rag3_24_app:app --reload
# - FastAPI 서비스: /search, 입력: 질의문, 출력: QA + 요약 결과 + 출처 정보
# - 1)  [사용자 질의]
# - 2)  [FastAPI 엔드포인트: /search]
# - 3)  [SentenceTransformer: 임베딩]
# - 4)  [Qdrant 의미 기반 검색]
# - 5)  [검색 결과 문서]
# - 6)  [KoELECTRA QA 모델 + MeCab 후처리(형태소 분석: 한국어 처리 강황)]
# - 7)  [KoBART Summarization 모델 + clean_summary]
# - 8)  [응답 + 출처 표시]
# - 9)  [최종 응답 LLM 모델은  외부 서비스 연계 검토: 자연스러운 문장]
# - 10) [최종 사용자 응답]

In [12]:
# 데이터 수집 모듈 구현
import feedparser               # RSS 피드를 파싱해서 뉴스 항목을 가져옴
from bs4 import BeautifulSoup   # HTML 태그 제거 및 텍스트 정제
from datetime import datetime
import psycopg2                 # PostgreSQL DB 연결 및 SQL 실행
import logging                  # 실행 과정 기록 (파일 + 콘솔 출력)

# RSS 가져오기 함수
def fetch_rss(rss_url: str):
    # feedparser 파싱 -> 뉴스 항목(entries) 리스트 반환
    feed = feedparser.parse(rss_url)
    return feed.entries

# 뉴스 파싱 함수
def parse_entry(entry):
    title = entry.title # 뉴스 제목, RSS 항목의 <title> 태그에 해당
    content_raw = entry.description if "description" in entry else ""  # 뉴스 본문, RSS 항목에 description 속성, 없으면 빈 문자열 반환
    content = BeautifulSoup(content_raw, "html.parser").get_text() # 텍스트만 추출, HTML 제거
    url = entry.link # 뉴스 원문 링크(URL), RSS 항목의 <link> 태그
    published_at = datetime(*entry.published_parsed[:6]) if hasattr(entry, "published_parsed") else datetime.now() # entry.published_parsed가 있으면 datetime 객체로 변환(연, 월, 일, 시, 분, 초)
    source_name = entry.source.title if hasattr(entry, "source") else "Google News" # entry.source가 있으면 그 안의 title을 사용하고, 없으면 기본값 "Google News"로 설정
    source_url = entry.source.href if hasattr(entry, "source") else "" # entry.source가 있으면 href 속성을 사용하고, 없으면 빈 문자열로 처리

    return title, content, url, published_at, source_name, source_url

# DB Insert 함수
def insert_article(cur, conn, title, content, url, published_at, source_name, source_url):
    try: # SQL 실행
        cur.execute("""
            INSERT INTO news_articles (title, content, url, published_at, source_name, source_url)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING;
        """, (title, content, url, published_at, source_name, source_url))
        if cur.rowcount > 0: # cur.rowcount → SQL 실행 후 영향을 받은 행 수
            logging.info("데이터 Insert 성공: %s", title) # 1 이상이면 새로운 데이터가 추가된 것 → “Insert 성공” 로그 기록
        else:
            logging.info("중복으로 건너뜀: %s", title) # 0이면 중복으로 인해 추가되지 않은 것 → “중복으로 건너뜀” 로그 기록
        conn.commit() # 실행 결과를 실제 DB 반영
    except Exception as e:
        logging.error("Insert 실패: %s, 에러: %s", title, e)

# 로깅 설정: 파일 + 콘솔 출력
logging.basicConfig(
    level=logging.INFO, # INFO 이상 레벨 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 파일 저장
        logging.StreamHandler() # 콘솔 출력
    ]
)
    
# RSS 함수 호출
rss_url = "https://news.google.com/rss?hl=ko&gl=KR&ceid=KR:ko"
entries = fetch_rss(rss_url)

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 뉴스 파싱 함수 -> DB Insert 함수
for entry in entries:
    title, content, url, published_at, source_name, source_url = parse_entry(entry)
    insert_article(cur, conn, title, content, url, published_at, source_name, source_url)

cur.close()
conn.close()
logging.info("DB 연결 종료 완료")

2026-05-13 11:58:07,792 [INFO] 데이터 Insert 성공: 방미 안규백 “‘호르무즈 정상화 단계적 기여 검토’ 뜻 미국에 전달” - 한겨레
2026-05-13 11:58:07,797 [INFO] 데이터 Insert 성공: 안철수 “기업 수익 국가가 나누는 건 공산주의” - 경향신문
2026-05-13 11:58:07,802 [INFO] 데이터 Insert 성공: 조현 "비행체 발사할 주체, 이란만 해도 여럿...민병대 가능성도" - 조선일보
2026-05-13 11:58:07,805 [INFO] 데이터 Insert 성공: “주왕산 사망 초등생, 추락에 의한 손상”…경찰 1차 소견, 빈소는 대구 - 한겨레
2026-05-13 11:58:07,809 [INFO] 데이터 Insert 성공: 트럼프 “이란 제안 쓰레기…휴전, 사망 직전 상태” 공격 재개하나 - 한겨레
2026-05-13 11:58:07,815 [INFO] 데이터 Insert 성공: “미국·이란 휴전 깨지면 UAE 가장 먼저 전쟁 휘말릴 듯” - KBS 뉴스
2026-05-13 11:58:07,828 [INFO] 데이터 Insert 성공: 구윤철 부총리 "삼성전자 파업 절대 안 돼…협상으로 해결되도록 지원할 것" - 조선일보
2026-05-13 11:58:07,832 [INFO] 중복으로 건너뜀: [단독]“지잡대 나왔냐” “어머니 공장서 일해”···‘갑질’ 일삼은 용산구의회 전문위원 - 경향신문
2026-05-13 11:58:07,838 [INFO] 데이터 Insert 성공: [속보] 오세훈vs정원오 오차범위 내 접전 “좁혀지는 격차”-여론조사공정 - 문화일보
2026-05-13 11:58:07,851 [INFO] 중복으로 건너뜀: “‘감사의 정원’ 자부심 뿜뿜” 오세훈…“장동혁과 전략적 역할 분담” - 한겨레
2026-05-13 11:58:07,855 [INFO] 데이터 Insert 성공: 사우디, 이란 본토 비밀 타격…'숨겨진 전선' 드러났다 - 마켓인
2026-05-13 1

In [13]:
# 뉴스 조회
import psycopg2
import logging

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 1. 전체 뉴스 개수 조회
try:
    cur.execute("""
            SELECT COUNT(*) 
            FROM news_articles;
    """)
    count = cur.fetchone()[0]
    logging.info("총 뉴스 개수: %s", count)
except Exception as e:
    logging.error("총 뉴스 개수 중 에러 발생:", e)

# 2. 최신 뉴스 5개 조회
try:
    cur.execute("""
            SELECT title, published_at
            FROM news_articles
            ORDER BY published_at DESC
            LIMIT 5;
    """)
    lastest_news = cur.fetchall()
    logging.info("최신 뉴스 5개:")
    for row in lastest_news:
        logging.info("제목: %s | 발행일: %s", row[0], row[1])
except Exception as e:
    logging.error("최신 뉴스 검색 중 에러 발생:", e)

# 3. 특정 키워드 검색(예시: AI)
try:
    keyword = 'AI'
    cur.execute("""
                SELECT title, url
                FROM news_articles
                WHERE content LIKE %s
                ORDER BY published_at DESC
                LIMIT 10;
    """, (f"%{keyword}%",)) # 튜플 형식 지켜야 에러가 안남
    keyword_news = cur.fetchall()
    logging.info("키워드 '%s' 관련 뉴스:", keyword)
    for row in keyword_news:
        logging.info("제목: %s | URL: %s", row[0], row[1])
except Exception as e:
    logging.error("키워드 검새 중 에러 발생:", e)

# 4. 출처별 뉴스 개수
try:
    cur.execute(""" 
            SELECT source_name, COUNT(*)
            FROM news_articles
            GROUP BY source_name
            ORDER BY COUNT(*) DESC;
    """)
    source_stats = cur.fetchall()
    logging.info("출처별 뉴스 개수:")
    for row in source_stats:
        logging.info("출처: %s | 개수: %s", row[0], row[1])
except Exception as e:
    logging.error("출처별 뉴스 개수 검색 중 에러 발생:", e)

cur.close()
conn.close()
logging.info("조회 완료 및 DB 연결 종료")

2026-05-13 11:58:16,887 [INFO] 총 뉴스 개수: 301
2026-05-13 11:58:16,902 [INFO] 최신 뉴스 5개:
2026-05-13 11:58:16,903 [INFO] 제목: ‘절단’ 대신 ‘재건’… 명지병원, 당뇨발 ‘비절단 미세재건’ 치료 강화 - 후생신보 | 발행일: 2026-05-13 02:46:00
2026-05-13 11:58:16,904 [INFO] 제목: “사상 첫 2회 연속 월드컵 16강을 향해”… 홍명보號 1400m 고지대서 막판 담금질 - 문화일보 | 발행일: 2026-05-13 02:18:07
2026-05-13 11:58:16,906 [INFO] 제목: 미·중 '호르무즈 통행료 반대' 합의…정상회담 논의 주목 - nocutnews.co.kr | 발행일: 2026-05-13 02:14:00
2026-05-13 11:58:16,907 [INFO] 제목: 트럼프 “이란 제안 쓰레기…휴전, 사망 직전 상태” 공격 재개하나 - 한겨레 | 발행일: 2026-05-13 02:05:00
2026-05-13 11:58:16,917 [INFO] 제목: 방미 안규백 “‘호르무즈 정상화 단계적 기여 검토’ 뜻 미국에 전달” - 한겨레 | 발행일: 2026-05-13 02:04:00
2026-05-13 11:58:16,936 [INFO] 키워드 'AI' 관련 뉴스:
2026-05-13 11:58:16,937 [INFO] 제목: 안철수 “기업 수익 국가가 나누는 건 공산주의” - 경향신문 | URL: https://news.google.com/rss/articles/CBMiWkFVX3lxTE1SMkZCVlQwaWRlR1B0OHgxUUZXRnpFUDVRT2g4VmYyeFNCYTFIOXdhSVRkclVIZ3dGQy04OVBhakI1Q3lxOWtqSlFJVlo3N1BkNG1lVmZxR1MzUdIBX0FVX3lxTFBPd3RwZUlENDJyb3pqWmdiU2k3bXlVXzFqX2dxVjdwSzlyQUdELWZIeDVXOGhRUTd

In [14]:
#  색인용 데이터 추출
import psycopg2
import logging

# DB 연결
conn = psycopg2.connect(
    dbname="newsdb", 
    user="newsuser", 
    password="1234", 
    host="localhost", 
    port="5432"
)

# cursor 객체 생성
cur = conn.cursor()

try:
    # 색인용 데이터 조회, 최근 데이터 100개 기준
    cur.execute(""" 
                SELECT id, title, content, url, published_at, source_name
                FROM news_articles
                ORDER BY published_at DESC
                LIMIT 100;
    """)
    articles = cur.fetchall()
    logging.info("색인용 데이터 %s개 가져옴:", len(articles))

    # Qdrant에 넣을 준비: 리스트 형태로 변환
    index_data = []
    for row in articles:
        record = {
            "id": row[0],
            "title": row[1],
            "content": row[2],
            "url": row[3],
            "published_at": row[4],
            "source_name": row[5]
        }
        index_data.append(record)

except Exception as e:
    logging.error("색인용 조회 에러 발생: %s", e)

# DB connect 종료
cur.close()
conn.close()
logging.info("색인 데이터 조회 완료 및 DB 연결 종료")

2026-05-13 11:58:38,329 [INFO] 색인용 데이터 100개 가져옴:
2026-05-13 11:58:38,333 [INFO] 색인 데이터 조회 완료 및 DB 연결 종료


In [15]:
# Qdrant Collection 생성
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import logging

# Qdrant 연결
qdrant = QdrantClient(host="localhost", port=6333)

# Qdrant Collection 초기화 + 생성
# qdrant.recreate_collection( # 지정한 이름의 컬렉션을 새로 생성, 이미 같은 이름의 컬렉션이 있으면 삭제 후 다시 생성(초기화 + 생성 기능)
#     collection_name="news_articles", # 컬렉션 명
#     vectors_config=VectorParams( # 벡터 설정 정의
#         size=384, # 모델 MiniLM-L12-v2 임베딩 차원 수(384차원)
#         distance=Distance.COSINE # 벡터 유사도 계산 방식(COSINE 추천, 의미 유사도 검색에 가장 많이 사용), DOT(내적 기반), EUCLID(거기 기반)
#     )
# )

# Qdrant Collecton 생성
if not qdrant.collection_exists("news_articles"):
    qdrant.create_collection(
        collection_name="news_articles", # 컬렉션 명
        vectors_config=VectorParams( # 벡터 설정 정의
            size=384, # 모델 MiniLM-L12-v2 임베딩 차원 수(384차원)
            distance=Distance.COSINE # 벡터 유사도 계산 방식(COSINE 추천, 의미 유사도 검색에 가장 많이 사용), DOT(내적 기반), EUCLID(거기 기반)
        )
    )
    logging.info(f"Qdrant Collection 'news_articles' 생성 완료")
else:
    logging.info(f"Qdrant Collection 'news_articles' 존재 합니다.")

2026-05-13 11:58:47,579 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-05-13 11:58:47,653 [INFO] HTTP Request: GET http://localhost:6333/collections/news_articles/exists "HTTP/1.1 200 OK"
2026-05-13 11:58:47,656 [INFO] Qdrant Collection 'news_articles' 존재 합니다.


In [ ]:
# Qdrant news_articles 컬렉션 update/insert
import torch
from sentence_transformers import SentenceTransformer # Hugging Face sentence-transformer 라이브러리 사용
from qdrant_client import QdrantClient
from qdrant_client.http import models # qdrant 컬렉션 및 검색 관련 설정 정의하는 데이터 모델(구조체) 로드
import logging

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
logging.info(f"Pytorch Version : {torch.__version__}, Device : {device}")

# 다국어 임베딩 모델 로드
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Qdrant 서버 연결
qdrant = QdrantClient(host="localhost", port=6333)

# 임베딩 함수 생성
def generate_embedding(text: str):
    # 본문(content)을 벡터로 변환
    embedding = model.encode(text)
    return embedding.tolist() # Qdrant에 넣기 위해 list 변환

# 데이터 저장 형식 설명
# qdrant.upsert()는 삽입(insert)+업데이트(update) 기능을 동시에 수행, 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가한다
    # { 
    #     "id": 1,
    #     "vector": [0.123, -0.456, 0.789, ...],
    #     "payload": {
    #         "title": "AI 의료 활용",
    #         "content": "AI가 의료 분야에서 활용되는 사례...",
    #         "date": "2026-03-11",
    #         "author": "홍길동"
    #     }
    # }
# Qdrant update/insert 함수
def insert_to_qdrant(data):
    ids = []
    vectors = []
    payloads = []
    
    for item in data:
        # 제목 + 내용 임베딩(벡터 변환) 대상 데이터
        text = item["title"] + " " + item["content"]
        # 임베딩 함수 호출
        embedding = generate_embedding(text=text)

        ids.append(item["id"]) # ID 값
        vectors.append(embedding) # 임베딩 처리 후 벡터 값
        payloads.append(item) # 원본 데이터
    
    qdrant.upsert(
        collection_name="news_articles",
        points=models.Batch(
            ids=ids,
            vectors=vectors,
            payloads=payloads # paylods(JSON) 형태로 저장
        )
    )
    logging.info("데이터 임베딩 및 Qdrant 저장 완료")
    

# Batch 단위로 Qdrant update/insert
batch_size = 2
total = len(index_data)

for i in range(0, len(index_data), batch_size):
    chunk = index_data[i : i + batch_size] # 2개씩 잘라서 가져온다
    insert_to_qdrant(chunk) # 잘라낸 청크를 Qdrant에 전달

    # 진행 상황 계산
    start_id = i
    end_id = i + len(chunk) - 1
    processed = end_id + 1
    progress = (processed / total) * 100
    remaining = total - processed

    logging.info(
        f"Batch {start_id} ~ {end_id} 인덱싱 완료" 
        f"(총 {len(chunk)}건), 진행률 : {progress:.2f}%, 남은 데이터 : {remaining}건"
    )

# 최종 데이터 개수 확인
count_result = qdrant.count(collection_name="news_articles")
logging.info(f"Qdrant news_articles 컬렉션 최종 저장 건수 : {count_result}건")


2026-05-13 12:04:21,499 [INFO] Pytorch Version : 2.2.2, Device : cpu
2026-05-13 12:04:21,520 [INFO] Use pytorch device_name: cpu
2026-05-13 12:04:21,522 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
2026-05-13 12:04:35,718 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-13 12:04:36,565 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles/points?wait=true "HTTP/1.1 200 OK"
2026-05-13 12:04:36,568 [INFO] 데이터 임베딩 및 Qdrant 저장 완료
2026-05-13 12:04:36,570 [INFO] Batch 0 ~ 1 인덱싱 완료(총 2건), 진행률 : 2.00%, 남은 데이터 : 98건


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-13 12:04:53,386 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles/points?wait=true "HTTP/1.1 200 OK"
2026-05-13 12:04:53,389 [INFO] 데이터 임베딩 및 Qdrant 저장 완료
2026-05-13 12:04:53,391 [INFO] Batch 96 ~ 97 인덱싱 완료(총 2건), 진행률 : 98.00%, 남은 데이터 : 2건


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-13 12:04:53,643 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles/points?wait=true "HTTP/1.1 200 OK"
2026-05-13 12:04:53,645 [INFO] 데이터 임베딩 및 Qdrant 저장 완료
2026-05-13 12:04:53,646 [INFO] Batch 98 ~ 99 인덱싱 완료(총 2건), 진행률 : 100.00%, 남은 데이터 : 0건


In [9]:
# # 임베딩 생성 후 Qdrant Collection에 Insert/Update
# # - 임베딩 모델 로드: 온라인, 오프라인 사용
# # - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# # 디바이스 설정
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# print(f"PyTorch Version: {torch.__version__}, Device: {device}")

# # 임베딩 모델 로드
# # - 여기서는 paraphrase-multilingual-MiniLM-L12-v2 모델을 사용했는데, 다국어(한국어 포함) 문장 의미를 잘 반영하는 임베딩을 생성한다
# # - 문장을 입력하면 의미 공간에서 가까운 벡터로 변환
# embedder = SentenceTransformer( # 온라인 상태(단, 로컬캐시에 저장)
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     device=device
# )

# # 오프라인 사용 준비: 온라인 환경에서 모델 다운로드
# # - git lfs install
# # - git clone https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# # embedder = SentenceTransformer( # 오프라인 상태(경로 지정에 저장)
# #     "./offline_models/paraphrase-multilingual-MiniLM-L12-v2",
# #     device=device
# # )


# # 컬렉션에 데이터 삽입
# def insert_to_qdrant(data):
#     ids = []
#     vectors = []
#     payloads = []
#     for item in data:
#         text = item["title"] + " " + item["content"] # 제목+내용 임베딩 대상 데이터
#         embedding = embedder.encode(text).tolist() # 제목+내용 임베딩

#         ids.append(item["id"]) # ID 값
#         vectors.append(embedding) # 벡터 데이터
#         payloads.append(item) # 원본 데이터
    
#     # qdrant.upsert()는 삽입(insert)+업데이트(update) 기능을 동시에 수행, 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가한다
#     # { - 데이터 저장 형식
#     #     "id": 1,
#     #     "vector": [0.123, -0.456, 0.789, ...],
#     #     "payload": {
#     #         "title": "AI 의료 활용",
#     #         "content": "AI가 의료 분야에서 활용되는 사례...",
#     #         "date": "2026-03-11",
#     #         "author": "홍길동"
#     #     }
#     # }
#     qdrant.upsert(
#         collection_name="news_articles_collection",
#         points=models.Batch(
#             ids=ids,
#             vectors=vectors,
#             payloads=payloads # payloads(JSON) 형태로 저장
#         )
#     )
#     print("데이터 임베딩 및 Qdrant 저장 완료")

# # Batch 단위로 Qdrant의 update/insert 테스트
# batch_size = 50
# for i in range(0, len(db_data), batch_size):
#     chunk = db_data[i:i+batch_size] # 2개씩 잘라서 가져옴
#     insert_to_qdrant(chunk) # 잘라낸 청크를 Qdrant에 전달

In [8]:
# # Qdrant collection 정보 확인: 전체 조회 + 특정 ID 조회
# from qdrant_client import QdrantClient

# qdrant = QdrantClient(host="localhost", port=6333)

# # 컬렉션 정보: 컬렉션 생성시 설정된 정보 및 상태를 보여주는 메타데이터
# info = qdrant.get_collection("news_articles_collection")
# print(info)

# # 전체 데이터 조회: limit 지정
# points = qdrant.scroll(
#     collection_name="news_articles_collection",
#     limit=10 # 최대 10개 반환
# )
# print(f"전체 데이터 조회: limit 지정\n{points}")

# # 특정 ID 데이터 조회
# point = qdrant.retrieve(
#     collection_name="news_articles_collection",
#     ids=[350,360],
#     with_payload=True, # payload 정보 포함
#     with_vectors=False # vectors 정보 제외
# )
# print(f"특정 ID 데이터 조회:\n{point}")